In [2]:
!pip uninstall -y jax jaxlib shap opencv-python opencv-python-headless opencv-contrib-python pytensor rasterio tobler xarray-einstats
!pip install -q "numpy<2" node2vec scikit-learn

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: shap 0.51.0
Uninstalling shap-0.51.0:
  Successfully uninstalled shap-0.51.0
Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-python-headless 4.13.0.92
Uninstalling opencv-python-headless-4.13.0.92:
  Successfully uninstalled opencv-python-headless-4.13.0.92
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
Found existing installation: pytensor 2.38.2
Uninstalling pytensor-2.38.2:
  Successfully uninstalled pytensor-2.38.2
Found existing installation: rasterio 1.5.0
Uninstalling rasterio-1.5.0:
  Successfully un

In [1]:
!pip install -q node2vec

In [ ]:
import os
os.kill(os.getpid(), 9)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import networkx as nx
from node2vec import Node2Vec

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.metrics import classification_report

In [3]:
ARTIFACT_DIR = Path("/content/pharmgraph_pass1_artifacts")
OUT_DIR = ARTIFACT_DIR / "pass1_node2vec_lr"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_GRAPH_FILE = ARTIFACT_DIR / "train_graph_edges.csv"

TRAIN_PAIRS_FILE = ARTIFACT_DIR / "gene_drug_train_pairs_labeled.csv"
VAL_PAIRS_FILE   = ARTIFACT_DIR / "gene_drug_val_pairs_labeled.csv"
TEST_PAIRS_FILE  = ARTIFACT_DIR / "gene_drug_test_pairs_labeled.csv"

TRAIN_POS_FILE = ARTIFACT_DIR / "gene_drug_train_pos.csv"
VAL_POS_FILE   = ARTIFACT_DIR / "gene_drug_val_pos.csv"
TEST_POS_FILE  = ARTIFACT_DIR / "gene_drug_test_pos.csv"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [5]:
train_graph_edges = pd.read_csv(TRAIN_GRAPH_FILE)

train_pairs = pd.read_csv(TRAIN_PAIRS_FILE)
val_pairs   = pd.read_csv(VAL_PAIRS_FILE)
test_pairs  = pd.read_csv(TEST_PAIRS_FILE)

train_pos = pd.read_csv(TRAIN_POS_FILE)
val_pos   = pd.read_csv(VAL_POS_FILE)
test_pos  = pd.read_csv(TEST_POS_FILE)

print("train_graph_edges:", train_graph_edges.shape)
print("train_pairs:", train_pairs.shape)
print("val_pairs:", val_pairs.shape)
print("test_pairs:", test_pairs.shape)

display(train_graph_edges.head())
display(train_pairs.head())

train_graph_edges: (68122, 6)
train_pairs: (103070, 6)
val_pairs: (17258, 6)
test_pairs: (17684, 6)


,relation_type,source_id,target_id,source_type,target_type,split
0,gene_disease,gene:nqo1,disease:pa151958383,gene,disease,NaN
1,gene_disease,gene:nqo1,disease:pa164925725,gene,disease,NaN
2,gene_disease,gene:nqo1,disease:pa166122058,gene,disease,NaN
3,gene_disease,gene:nqo1,disease:pa166123026,gene,disease,NaN
4,gene_disease,gene:nqo1,disease:pa166123431,gene,disease,NaN


,source_id,source_raw,target_id,target_raw,split,label
0,gene:pltp,NaN,"drug:[ser11](n-cnp,c-anp)pbnp2-15",NaN,NaN,0
1,gene:b2m,NaN,drug:coagulation factor viia (recombinant)-jncw,NaN,NaN,0
2,gene:csh2,NaN,drug:chembl:chembl1256984,NaN,NaN,0
3,gene:xk,xk,drug:bupivacaine,bupivacaine,train,1
4,gene:nedd4l,NaN,drug:tetradifon,NaN,NaN,0


In [6]:
G = nx.Graph()

for _, row in train_graph_edges.iterrows():
    s = row["source_id"]
    t = row["target_id"]
    rel = row["relation_type"]

    # unweighted baseline graph
    if G.has_edge(s, t):
        continue
    G.add_edge(s, t, relation_type=rel)

print("Graph summary")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Graph summary
Nodes: 27355
Edges: 68122


In [7]:
from gensim.models.callbacks import CallbackAny2Vec

class EpochLogger(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        print(f"Epoch {self.epoch + 1} started")

    def on_epoch_end(self, model):
        print(f"Epoch {self.epoch + 1} finished")
        self.epoch += 1

EMBED_DIM = 128
WALK_LENGTH = 40
NUM_WALKS = 10
WINDOW = 10
MIN_COUNT = 1
BATCH_WORDS = 256
WORKERS = 4
EPOCHS = 5   # set explicitly so you can see progress by epoch

node2vec = Node2Vec(
    G,
    dimensions=EMBED_DIM,
    walk_length=WALK_LENGTH,
    num_walks=NUM_WALKS,
    workers=WORKERS,
    seed=RANDOM_SEED,
    quiet=False   # lets walk-generation progress show
)

epoch_logger = EpochLogger()

n2v_model = node2vec.fit(
    window=WINDOW,
    min_count=MIN_COUNT,
    batch_words=BATCH_WORDS,
    epochs=EPOCHS,
    callbacks=[epoch_logger]
)

print("Node2Vec training complete.")

Computing transition probabilities:   0%|          | 0/27355 [00:00<?, ?it/s]

Epoch 1 started
Epoch 1 finished
Epoch 2 started
Epoch 2 finished
Epoch 3 started
Epoch 3 finished
Epoch 4 started
Epoch 4 finished
Epoch 5 started
Epoch 5 finished
Node2Vec training complete.


In [8]:
embedding_rows = []
for node in G.nodes():
    vec = n2v_model.wv[node]
    row = {"node_id": node}
    for i, val in enumerate(vec):
        row[f"emb_{i}"] = float(val)
    embedding_rows.append(row)

embeddings_df = pd.DataFrame(embedding_rows)
embeddings_df.to_csv(OUT_DIR / "node2vec_embeddings.csv", index=False)

print("Saved:", OUT_DIR / "node2vec_embeddings.csv")
display(embeddings_df.head())

Saved: /content/pharmgraph_pass1_artifacts/pass1_node2vec_lr/node2vec_embeddings.csv


,node_id,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,...,emb_118,emb_119,emb_120,emb_121,emb_122,emb_123,emb_124,emb_125,emb_126,emb_127
0,gene:nqo1,0.384057,-0.164343,0.635956,0.118586,0.092257,0.142183,-0.357883,0.075543,0.127317,...,0.414448,0.381079,0.061358,-0.282320,-0.205283,0.361829,-0.239368,0.079614,-0.170142,0.563075
1,disease:pa151958383,0.305444,-0.308869,0.427510,0.311362,-0.109404,0.051316,0.331288,0.051812,0.333468,...,-0.071189,0.147814,-0.141646,0.041979,-0.033615,0.381729,-0.103226,-0.215702,0.299704,0.457679
2,disease:pa164925725,0.242227,0.112500,0.164388,-0.404677,0.400232,-0.176820,0.269950,-0.285765,-0.143979,...,0.533472,-0.157379,0.036309,-0.436105,-0.339742,0.099625,-0.125872,0.222219,0.098677,0.348811
3,disease:pa166122058,0.096136,-0.022354,0.285299,0.265939,0.456294,-0.137375,-0.008318,-0.222436,-0.002149,...,-0.004496,0.159429,0.062375,-0.059938,-0.016204,0.663379,-0.169251,0.250946,-0.240638,0.295554
4,disease:pa166123026,-0.283094,-0.284931,0.084682,0.026790,0.026407,-0.059574,0.033305,0.193239,0.023475,...,-0.159452,0.028444,-0.264673,-0.114865,-0.371315,0.348859,0.216748,-0.156780,-0.083617,0.013513


In [9]:
embedding_map = {node: n2v_model.wv[node] for node in G.nodes()}

def build_pair_features(pairs_df, embedding_map, embedding_dim=128):
    """
    Build pairwise features for gene-drug pairs.
    Returns:
      X: np.ndarray
      y: np.ndarray
      valid_pairs_df: dataframe of successfully featurized pairs
      dropped_df: dataframe of pairs that could not be featurized
    """
    features = []
    labels = []
    valid_rows = []
    dropped_rows = []

    for _, row in pairs_df.iterrows():
        g = row["source_id"]
        d = row["target_id"]
        y = row["label"]

        g_vec = embedding_map.get(g)
        d_vec = embedding_map.get(d)

        if g_vec is None or d_vec is None:
            dropped_rows.append(row.to_dict())
            continue

        prod = g_vec * d_vec
        diff = np.abs(g_vec - d_vec)

        feat = np.concatenate([g_vec, d_vec, prod, diff])

        features.append(feat)
        labels.append(y)
        valid_rows.append(row.to_dict())

    X = np.vstack(features) if features else np.empty((0, embedding_dim * 4))
    y = np.array(labels, dtype=int)
    valid_pairs_df = pd.DataFrame(valid_rows)
    dropped_df = pd.DataFrame(dropped_rows)

    return X, y, valid_pairs_df, dropped_df

In [10]:
X_train, y_train, train_pairs_valid, train_pairs_dropped = build_pair_features(
    train_pairs, embedding_map, embedding_dim=EMBED_DIM
)

X_val, y_val, val_pairs_valid, val_pairs_dropped = build_pair_features(
    val_pairs, embedding_map, embedding_dim=EMBED_DIM
)

X_test, y_test, test_pairs_valid, test_pairs_dropped = build_pair_features(
    test_pairs, embedding_map, embedding_dim=EMBED_DIM
)

print("Feature matrix shapes")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

print("\nDropped pairs")
print("train dropped:", len(train_pairs_dropped))
print("val dropped:  ", len(val_pairs_dropped))
print("test dropped: ", len(test_pairs_dropped))

Feature matrix shapes
X_train: (103070, 512) y_train: (103070,)
X_val:   (17258, 512) y_val:   (17258,)
X_test:  (17684, 512) y_test:  (17684,)

Dropped pairs
train dropped: 0
val dropped:   0
test dropped:  0


In [11]:
lr = LogisticRegression(
    random_state=RANDOM_SEED,
    max_iter=1000,
    class_weight="balanced"
)

lr.fit(X_train, y_train)
print("Logistic regression training complete.")

Logistic regression training complete.


In [12]:
val_probs = lr.predict_proba(X_val)[:, 1]
test_probs = lr.predict_proba(X_test)[:, 1]

val_preds = (val_probs >= 0.5).astype(int)
test_preds = (test_probs >= 0.5).astype(int)

val_roc_auc = roc_auc_score(y_val, val_probs)
val_ap = average_precision_score(y_val, val_probs)

test_roc_auc = roc_auc_score(y_test, test_probs)
test_ap = average_precision_score(y_test, test_probs)

print("Validation metrics")
print("ROC-AUC:", round(val_roc_auc, 4))
print("Average Precision:", round(val_ap, 4))

print("\nTest metrics")
print("ROC-AUC:", round(test_roc_auc, 4))
print("Average Precision:", round(test_ap, 4))

Validation metrics
ROC-AUC: 0.9355
Average Precision: 0.9496

Test metrics
ROC-AUC: 0.9353
Average Precision: 0.9503


In [13]:
def precision_recall_at_k(y_true, y_score, k):
    order = np.argsort(-y_score)
    y_true_sorted = np.array(y_true)[order]

    topk = y_true_sorted[:k]
    precision_k = topk.mean() if len(topk) > 0 else 0.0

    total_positives = np.sum(y_true_sorted)
    recall_k = np.sum(topk) / total_positives if total_positives > 0 else 0.0

    return precision_k, recall_k

k_values = [10, 25, 50, 100, 250, 500]

metric_rows = []

for split_name, y_true, y_score, roc_auc, ap in [
    ("val", y_val, val_probs, val_roc_auc, val_ap),
    ("test", y_test, test_probs, test_roc_auc, test_ap),
]:
    row = {
        "split": split_name,
        "roc_auc": roc_auc,
        "average_precision": ap,
    }

    for k in k_values:
        p_at_k, r_at_k = precision_recall_at_k(y_true, y_score, k)
        row[f"precision@{k}"] = p_at_k
        row[f"recall@{k}"] = r_at_k

    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

,split,roc_auc,average_precision,precision@10,recall@10,precision@25,recall@25,precision@50,recall@50,precision@100,recall@100,precision@250,recall@250,precision@500,recall@500
0,val,0.935473,0.949558,1.0,0.001159,1.0,0.002897,1.0,0.005794,1.0,0.011589,1.0,0.028972,0.998,0.057828
1,test,0.935336,0.950261,1.0,0.001131,1.0,0.002827,1.0,0.005655,1.0,0.011310,1.0,0.028274,1.000,0.056548


In [14]:
print("Validation classification report")
print(classification_report(y_val, val_preds, digits=4))

print("Test classification report")
print(classification_report(y_test, test_preds, digits=4))

Validation classification report
              precision    recall  f1-score   support

           0     0.7132    0.9912    0.8295      8629
           1     0.9856    0.6013    0.7469      8629

    accuracy                         0.7963     17258
   macro avg     0.8494    0.7963    0.7882     17258
weighted avg     0.8494    0.7963    0.7882     17258

Test classification report
              precision    recall  f1-score   support

           0     0.7088    0.9925    0.8270      8842
           1     0.9876    0.5923    0.7405      8842

    accuracy                         0.7924     17684
   macro avg     0.8482    0.7924    0.7838     17684
weighted avg     0.8482    0.7924    0.7838     17684



In [15]:
val_scored = val_pairs_valid.copy()
val_scored["pred_prob"] = val_probs
val_scored["pred_label_0.5"] = val_preds

test_scored = test_pairs_valid.copy()
test_scored["pred_prob"] = test_probs
test_scored["pred_label_0.5"] = test_preds

val_scored.to_csv(OUT_DIR / "val_scored_predictions.csv", index=False)
test_scored.to_csv(OUT_DIR / "test_scored_predictions.csv", index=False)

metrics_df.to_csv(OUT_DIR / "baseline_metrics.csv", index=False)

print("Saved:")
print(OUT_DIR / "val_scored_predictions.csv")
print(OUT_DIR / "test_scored_predictions.csv")
print(OUT_DIR / "baseline_metrics.csv")

Saved:
/content/pharmgraph_pass1_artifacts/pass1_node2vec_lr/val_scored_predictions.csv
/content/pharmgraph_pass1_artifacts/pass1_node2vec_lr/test_scored_predictions.csv
/content/pharmgraph_pass1_artifacts/pass1_node2vec_lr/baseline_metrics.csv


In [16]:
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

lr_final = LogisticRegression(
    random_state=RANDOM_SEED,
    max_iter=1000,
    class_weight="balanced"
)

lr_final.fit(X_trainval, y_trainval)
print("Final LR model trained on train+val.")

Final LR model trained on train+val.


In [17]:
all_train_nodes = sorted(G.nodes())

all_genes = sorted([n for n in all_train_nodes if n.startswith("gene:")])
all_drugs = sorted([n for n in all_train_nodes if n.startswith("drug:")])

known_positive_pairs = set(
    map(tuple,
        pd.concat([train_pos, val_pos, test_pos], ignore_index=True)[["source_id", "target_id"]].to_records(index=False)
    )
)

print("Genes in graph:", len(all_genes))
print("Drugs in graph:", len(all_drugs))
print("Known positive pairs:", len(known_positive_pairs))

Genes in graph: 5292
Drugs in graph: 18481
Known positive pairs: 69006


In [18]:
candidate_pairs = []

for g in all_genes:
    for d in all_drugs:
        pair = (g, d)
        if pair in known_positive_pairs:
            continue
        candidate_pairs.append(pair)

candidate_df = pd.DataFrame(candidate_pairs, columns=["source_id", "target_id"])
candidate_df["label"] = -1

print("Candidate unseen pairs:", len(candidate_df))
display(candidate_df.head())

Candidate unseen pairs: 97732446


,source_id,target_id,label
0,gene:a12m1,drug:&alpha;&beta;-meatp,-1
1,gene:a12m1,drug:&alpha;&beta;-methyleneadp,-1
2,gene:a12m1,drug:&alpha;-bungarotoxin,-1
3,gene:a12m1,drug:&alpha;-cgrp,-1
4,gene:a12m1,drug:&alpha;-conotoxin arib,-1


In [19]:
def score_candidate_pairs_in_batches(candidate_df, embedding_map, model, embedding_dim=128, batch_size=50000):
    scored_parts = []

    for start in range(0, len(candidate_df), batch_size):
        end = min(start + batch_size, len(candidate_df))
        batch = candidate_df.iloc[start:end].copy()

        X_batch, _, valid_batch, dropped_batch = build_pair_features(
            batch.assign(label=0), embedding_map, embedding_dim=embedding_dim
        )

        if len(valid_batch) == 0:
            continue

        probs = model.predict_proba(X_batch)[:, 1]
        valid_batch = valid_batch.copy()
        valid_batch["pred_prob"] = probs

        scored_parts.append(valid_batch[["source_id", "target_id", "pred_prob"]])

        print(f"Scored {end:,} / {len(candidate_df):,}")

    if len(scored_parts) == 0:
        return pd.DataFrame(columns=["source_id", "target_id", "pred_prob"])

    return pd.concat(scored_parts, ignore_index=True)

candidate_scored = score_candidate_pairs_in_batches(
    candidate_df,
    embedding_map,
    lr_final,
    embedding_dim=EMBED_DIM,
    batch_size=50000
)

candidate_scored = candidate_scored.sort_values("pred_prob", ascending=False).reset_index(drop=True)
print("Scored candidate pairs:", candidate_scored.shape)
display(candidate_scored.head(20))

Scored 50,000 / 97,732,446
Scored 100,000 / 97,732,446
Scored 150,000 / 97,732,446
Scored 200,000 / 97,732,446
Scored 250,000 / 97,732,446
Scored 300,000 / 97,732,446
Scored 350,000 / 97,732,446
Scored 400,000 / 97,732,446
Scored 450,000 / 97,732,446
Scored 500,000 / 97,732,446
Scored 550,000 / 97,732,446
Scored 600,000 / 97,732,446
Scored 650,000 / 97,732,446
Scored 700,000 / 97,732,446
Scored 750,000 / 97,732,446
Scored 800,000 / 97,732,446
Scored 850,000 / 97,732,446
Scored 900,000 / 97,732,446
Scored 950,000 / 97,732,446
Scored 1,000,000 / 97,732,446
Scored 1,050,000 / 97,732,446
Scored 1,100,000 / 97,732,446
Scored 1,150,000 / 97,732,446
Scored 1,200,000 / 97,732,446
Scored 1,250,000 / 97,732,446
Scored 1,300,000 / 97,732,446
Scored 1,350,000 / 97,732,446
Scored 1,400,000 / 97,732,446
Scored 1,450,000 / 97,732,446
Scored 1,500,000 / 97,732,446
Scored 1,550,000 / 97,732,446
Scored 1,600,000 / 97,732,446
Scored 1,650,000 / 97,732,446
Scored 1,700,000 / 97,732,446
Scored 1,750,000 / 

,source_id,target_id,pred_prob
0,gene:actbp6,drug:mln3126,1.0
1,gene:il33,drug:spesolimab,1.0
2,gene:ccr9,drug:compound 24 [pmid: 26987013 ],1.0
3,gene:ccr9,drug:[125i]ccl25 (human),1.0
4,gene:il33,drug:imsidolimab,1.0
5,gene:il1rap,drug:etokimab,1.0
6,gene:il1rl2,drug:astegolimab,1.0
7,gene:il1rap,drug:itepekimab,1.0
8,gene:il1rap,drug:torudokimab,1.0
9,gene:il1rl1,drug:etokimab,1.0


In [20]:
TOP_K_EXPORT = 500

top_candidates = candidate_scored.head(TOP_K_EXPORT).copy()

top_candidates.to_csv(OUT_DIR / "top_predicted_novel_gene_drug_links.csv", index=False)

print("Saved:", OUT_DIR / "top_predicted_novel_gene_drug_links.csv")
display(top_candidates.head(50))

Saved: /content/pharmgraph_pass1_artifacts/pass1_node2vec_lr/top_predicted_novel_gene_drug_links.csv


,source_id,target_id,pred_prob
0,gene:actbp6,drug:mln3126,1.0
1,gene:il33,drug:spesolimab,1.0
2,gene:ccr9,drug:compound 24 [pmid: 26987013 ],1.0
3,gene:ccr9,drug:[125i]ccl25 (human),1.0
4,gene:il33,drug:imsidolimab,1.0
5,gene:il1rap,drug:etokimab,1.0
6,gene:il1rl2,drug:astegolimab,1.0
7,gene:il1rap,drug:itepekimab,1.0
8,gene:il1rap,drug:torudokimab,1.0
9,gene:il1rl1,drug:etokimab,1.0


In [21]:
experiment_summary = pd.DataFrame([
    {"item": "graph_nodes", "value": G.number_of_nodes()},
    {"item": "graph_edges", "value": G.number_of_edges()},
    {"item": "embedding_dim", "value": EMBED_DIM},
    {"item": "walk_length", "value": WALK_LENGTH},
    {"item": "num_walks", "value": NUM_WALKS},
    {"item": "window", "value": WINDOW},
    {"item": "train_pairs", "value": len(train_pairs_valid)},
    {"item": "val_pairs", "value": len(val_pairs_valid)},
    {"item": "test_pairs", "value": len(test_pairs_valid)},
    {"item": "val_roc_auc", "value": val_roc_auc},
    {"item": "val_average_precision", "value": val_ap},
    {"item": "test_roc_auc", "value": test_roc_auc},
    {"item": "test_average_precision", "value": test_ap},
])

experiment_summary.to_csv(OUT_DIR / "experiment_summary.csv", index=False)
display(experiment_summary)

,item,value
0,graph_nodes,27355.000000
1,graph_edges,68122.000000
2,embedding_dim,128.000000
3,walk_length,40.000000
4,num_walks,10.000000
5,window,10.000000
6,train_pairs,103070.000000
7,val_pairs,17258.000000
8,test_pairs,17684.000000
9,val_roc_auc,0.935473
